# 프로게스테론 De Novo 단백질 설계
### RFdiffusionAA 파이프라인

전체 파이프라인:

| 단계 | 상태 | 도구 | 내용 |
|------|------|------|------|
| Step 1 | 완료 | PyMOL | PDB 1A28 에서 STR 좌표 추출 |
| Step 2 | 이번 세션 | RFdiffusionAA | 프로게스테론 주변 backbone 생성 |
| Step 3 | 예정 | LigandMPNN | backbone -> 서열 설계 (30개) |
| Step 4 | 예정 | ColabFold | 서열 -> pAE 필터 -> 상위 3~5개 |
| Step 5 | 예정 | PyMOL | 포켓 시각화 + 수소결합 분석 |
| Step 6 | 예정 | GitHub | README + 포트폴리오 정리 |

---

실행 전 필수 확인: 상단 메뉴 [런타임 > 런타임 유형 변경 > T4 GPU]  
RFdiffusionAA는 딥러닝 모델이므로 GPU 없으면 실행 불가.

backbone 생성 전략: 1개씩 실행 -> 완료 즉시 Drive 저장 확인 -> 반복  
런타임이 끊겨도 완료된 것은 Drive에 보존됨.

---
## 1단계: rf_diffusion_all_atom 레포 클론

원본 WS10 코드와 동일한 방식으로 레포를 클론한다.  
run_inference.py 가 이 레포 안에 있기 때문에 반드시 필요하다.  

git clone: GitHub 레포지토리를 현재 환경에 통째로 복사하는 명령어  
%cd: Colab 작업 디렉토리를 변경. 이후 모든 명령어는 이 폴더 기준으로 실행됨  
git submodule: RFdiffusionAA가 의존하는 외부 라이브러리들을 함께 가져오는 명령어

In [ ]:
# 원본 WS10과 동일
!git clone https://github.com/baker-laboratory/rf_diffusion_all_atom.git
%cd rf_diffusion_all_atom

!git submodule init
!git submodule update

---
## 2단계: Singularity 환경 설정

원본 WS10은 .sif 파일을 wget으로 새로 다운로드(20분 이상 소요)했다.  
우리는 저번 세션에서 이미 Drive에 저장해뒀으므로 Drive에서 복사한다 (약 2분).  

환경 변수 설정 부분(LD_PRELOAD, APPTAINER_BINDPATH 등)은 원본과 동일하게 유지한다.  

Singularity (.sif): Docker와 유사한 컨테이너 기술. RFdiffusionAA 실행에 필요한  
모든 라이브러리가 하나의 파일에 패키징되어 있음. 크기: 12.67GB.  

Drive에 저장된 파일 구조:  
MyDrive/RFdiffusionAA/  
  rf_se3_diffusion.sif            (12.67GB)  
  RFDiffusionAA_paper_weights.pt  (1.25GB)  
  input/  
    1A28.pdb

In [ ]:
# 원본 WS10과 동일한 환경 변수 설정
import os

os.environ["LD_PRELOAD"] = ""
os.environ["APPTAINER_BINDPATH"] = "/content"
os.environ["LMOD_CMD"] = "/usr/share/lmod/lmod/libexec/lmod"

!curl -J -O https://raw.githubusercontent.com/NeuroDesk/neurocommand/main/googlecolab_setup.sh
!chmod +x googlecolab_setup.sh
!./googlecolab_setup.sh

os.environ["MODULEPATH"] = ':'.join(map(str, list(map(lambda x: os.path.join(os.path.abspath('/cvmfs/neurodesk.ardc.edu.au/neurodesk-modules/'), x), os.listdir('/cvmfs/neurodesk.ardc.edu.au/neurodesk-modules/')))))

!apptainer exec docker://alpine cat /etc/alpine-release
!singularity exec docker://alpine cat /etc/alpine-release

In [ ]:
# Drive에서 .sif 복사 (원본 WS10의 wget 다운로드를 Drive 복사로 대체)
# 이유: .sif가 이미 Drive에 있으므로 매번 다운로드할 필요 없음 (20분 -> 2분)
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/RFdiffusionAA'

# 현재 위치: /content/rf_diffusion_all_atom/
# .sif 와 가중치를 이 폴더 안으로 복사

print('1/3) .sif 복사 중... (약 2분 소요)')
shutil.copy(f'{DRIVE_BASE}/rf_se3_diffusion.sif', 'rf_se3_diffusion.sif')
print(f'     완료: {os.path.getsize("rf_se3_diffusion.sif")/1e9:.2f} GB')

print('2/3) 모델 가중치 복사 중...')
shutil.copy(f'{DRIVE_BASE}/RFDiffusionAA_paper_weights.pt', 'RFDiffusionAA_paper_weights.pt')
print(f'     완료: {os.path.getsize("RFDiffusionAA_paper_weights.pt")/1e9:.2f} GB')

print('3/3) 1A28.pdb 복사 중...')
os.makedirs('input', exist_ok=True)
shutil.copy(f'{DRIVE_BASE}/input/1A28.pdb', 'input/1A28.pdb')
print('     완료')

# 컨테이너 정상 동작 확인 (원본 WS10과 동일)
!singularity exec rf_se3_diffusion.sif cat /etc/os-release

---
## 3단계: RFdiffusionAA 실행

원본 WS10과 singularity run 명령어 구조는 완전히 동일하다.  
변경된 파라미터는 우리 프로젝트 타겟에 맞게 수정한 것뿐이다.

원본 WS10 파라미터 vs 우리 파라미터:

| 파라미터 | WS10 원본 | 우리 버전 | 변경 이유 |
|----------|-----------|-----------|----------|
| input_pdb | 7v11.pdb | 1A28.pdb | 프로게스테론+PR 복합체 구조 |
| ligand | OQO | STR | 프로게스테론 PDB 3글자 코드 |
| contigs | ['150-150'] | ['100-150'] | 길이 범위를 넓혀 다양성 확보 |
| num_designs | 1 | 1 | 1개씩 실행해서 완료 즉시 Drive 저장 확인 |
| design_startnum | 0 | 매 실행마다 변경 | 파일명 겹침 방지 |
| output_prefix | output/ligand_only/sample | Drive 경로 | 런타임 끊겨도 결과 보존 |

contigmap.contigs: 생성할 단백질의 길이 범위.  
['150-150'] = 정확히 150 아미노산, ['100-150'] = 100~150 중 랜덤.  
실제 랩에서는 넓게 주고 나중에 필터링하는 방식을 사용.  

inference.ligand: 입력 PDB 안의 어떤 리간드를 타겟으로 할지 지정.  
PDB 3글자 코드 사용. 프로게스테론 = STR.

design_startnum 규칙:  
1번째 실행: design_startnum=0 -> sample_0.pdb  
2번째 실행: design_startnum=1 -> sample_1.pdb  
3번째 실행: design_startnum=2 -> sample_2.pdb  
같은 번호로 돌리면 기존 파일이 덮어씌워지므로 주의.

In [ ]:
# 실행 시간 측정 시작 (원본 WS10과 동일)
import time
start_time = time.time()

In [ ]:
# Drive 출력 폴더 생성
# output_prefix를 Drive 경로로 지정 -> 완료 즉시 Drive에 저장됨
import os
OUTPUT_DIR = '/content/drive/MyDrive/RFdiffusionAA/output/progesterone'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# design_startnum: 이번에 생성할 파일 번호
# 1번째 실행: 0 -> sample_0.pdb
# 2번째 실행: 1 -> sample_1.pdb
# 3번째 실행: 2 -> sample_2.pdb
# num_designs=1 고정: 1개 완료 -> Drive 저장 확인 -> 다음 번호로 재실행
!singularity run --nv rf_se3_diffusion.sif -u run_inference.py inference.deterministic=True diffuser.T=100 inference.output_prefix=/content/drive/MyDrive/RFdiffusionAA/output/progesterone/sample inference.input_pdb=input/1A28.pdb contigmap.contigs=[\'100-150\'] inference.ligand=STR inference.num_designs=1 inference.design_startnum=0

In [ ]:
# 실행 시간 측정 종료 (원본 WS10과 동일)
end_time = time.time()
elapsed_time = end_time - start_time
print(f'실행 시간: {elapsed_time:.2f} 초 ({elapsed_time/60:.1f} 분)')

---
## 4단계: 결과 확인 및 다운로드

원본 WS10은 files.download()로 직접 다운로드했다.  
우리는 output_prefix를 Drive로 지정했으므로 이미 Drive에 저장되어 있다.  
files.download()도 그대로 사용 가능하다 (Drive 경로로 수정).

In [ ]:
# 셀 1: 결과 확인
# os.listdir()은 해당 폴더 직속 파일/폴더만 반환 -> traj/ 안까지는 안 들어감
# 따라서 'traj' not in f 필터 불필요 -> .pdb 로 끝나는지만 확인하면 충분
import os

OUTPUT_DIR = '/content/drive/MyDrive/RFdiffusionAA/output/progesterone'

print('생성된 파일 목록:')
files_list = sorted(os.listdir(OUTPUT_DIR))

if not files_list:
    print('  파일 없음 -> 실행이 완료되지 않았거나 에러 발생')
else:
    for f in files_list:
        full_path = f'{OUTPUT_DIR}/{f}'
        size_kb = os.path.getsize(full_path) / 1e3
        print(f'  {f}  ({size_kb:.1f} KB)')

# CA(알파탄소) 원자 수 = 잔기 수
# CA: 단백질 주쇄에서 각 아미노산마다 하나씩 있는 원자 -> CA 개수 = 아미노산 개수
print('\n각 backbone 잔기 수:')
pdb_files = [f for f in files_list if f.endswith('.pdb')]

if not pdb_files:
    # 방어 코드: 에러 없이 조용히 넘어가는 상황 방지
    print('  PDB 파일 없음 -> RFdiffusionAA 실행이 아직 안 끝났을 수 있음')
else:
    for f in pdb_files:
        path = f'{OUTPUT_DIR}/{f}'
        with open(path) as pf:
            n_ca = sum(1 for line in pf if line.startswith('ATOM') and ' CA ' in line)
        print(f'  {f}: {n_ca} 잔기')

In [ ]:
# 셀 2: sample pdb 다운로드
# Drive에 이미 저장되어 있으므로 필수는 아님
# 로컬 PyMOL에서 바로 열어보려면 실행
from google.colab import files
import os

OUTPUT_DIR = '/content/drive/MyDrive/RFdiffusionAA/output/progesterone'

downloaded = 0
for i in range(5):
    pdb_path = f'{OUTPUT_DIR}/sample_{i}.pdb'
    if os.path.exists(pdb_path):
        files.download(pdb_path)
        print(f'  다운로드: sample_{i}.pdb')
        downloaded += 1

if downloaded == 0:
    print('다운로드할 파일 없음 -> 경로 확인 필요')
else:
    print(f'\n총 {downloaded}개 다운로드 완료')

In [ ]:
# 셀 3: trajectory 파일 다운로드
# _Xt-1_traj.pdb: 각 denoising step의 중간 구조 (노이즈가 아직 남아있는 상태)
# _pX0_traj.pdb:  각 step에서 모델이 예측한 최종 구조 (매 step마다 현재 예측)
# 둘 다 PyMOL에서 애니메이션으로 diffusion 과정 시각화 가능 -> 포트폴리오 자료
from google.colab import files
import os

OUTPUT_DIR = '/content/drive/MyDrive/RFdiffusionAA/output/progesterone'
TRAJ_DIR = f'{OUTPUT_DIR}/traj'

if not os.path.exists(TRAJ_DIR):
    print('traj 폴더 없음 (버전에 따라 생성되지 않을 수 있음)')
else:
    downloaded = 0
    for i in range(5):
        xt1 = f'{TRAJ_DIR}/sample_{i}_Xt-1_traj.pdb'
        px0 = f'{TRAJ_DIR}/sample_{i}_pX0_traj.pdb'
        for path in [xt1, px0]:
            if os.path.exists(path):
                files.download(path)
                print(f'  다운로드: {os.path.basename(path)}')
                downloaded += 1
    print(f'\n총 {downloaded}개 trajectory 파일 다운로드 완료')

---
## 트러블슈팅 기록

| # | 에러/문제 | 원인 | 해결 |
|---|-----------|------|------|
| 1 | ValueError: max() arg is an empty sequence | progesterone_STR.pdb에 단백질 없음 -> idx_prot 빈 리스트 | 입력을 단백질+리간드 둘 다 있는 1A28.pdb로 변경 |
| 2 | Drive 복사 실패 (용량 부족) | .sif 12.67GB -> Drive 15GB 초과 | 새 Google 계정 생성 -> Drive 15GB 새로 할당 |
| 3 | 런타임 끊김으로 파일 소멸 | Colab 세션 자동 종료 | output_prefix를 Drive 경로로 설정 |
| 4 | torchaudio 충돌 경고 | torch 2.5.0 vs torchaudio 2.10.0 버전 불일치 | 무시. RFdiffusionAA는 torchaudio 사용 안 함 |
| 5 | GPU 표준 런타임 전환 경고 팝업 | Colab이 GPU 미사용 감지 | 무조건 취소. GPU 없으면 실행 불가 |
| 6 | IndentationError: unexpected indent | 백슬래시 멀티라인에 자동 들여쓰기 삽입 | singularity 명령어를 한 줄로 작성 |
| 7 | FileNotFoundError: 1A28.pdb | Drive input/ 폴더에 파일 없음 | RCSB에서 직접 다운로드 후 Drive 백업 |

---
## 다음 세션 (Step 3: LigandMPNN)

Drive에 저장된 sample_0.pdb ~ sample_N.pdb 를 입력으로 사용.  
각 backbone당 30개 서열 설계 -> backbone 3개면 총 90개.  
ColabFold pAE_interaction < 10 필터로 상위 3~5개 선별.  

진행 현황: Step 1 완료 | Step 2 이번 세션 | Step 3~6 예정